# Notes from AIMA Chapter 14 Probabilistic Reasoning Over Time


# Technical Report: Probabilistic Reasoning Over Time

## 1. Foundations of Temporal Probabilistic Models

### The Strategic Shift: From Snapshots to Movies

In static Bayesian networks, reasoning occurs at a single moment in time. Temporal models extend this idea so a rational agent can maintain an internal belief while receiving a continuous stream of percepts. The core questions become:

* What is true now?
* How did we get here?
* What happens next?

Temporal reasoning transforms inference into continuous belief tracking.

---

### The Anatomy of Time: States and Percepts

Time is represented as discrete slices $t = 0,1,2,\dots$.

| Variable Category  | Symbol                | Role                          |
| ------------------ | --------------------- | ----------------------------- |
| State Variables    | $X_t$                 | Hidden true state at time $t$ |
| Evidence Variables | $E_t$                 | Observed percepts at time $t$ |
| Transition Model   | $P(X_t \mid X_{t-1})$ | Dynamics or laws of motion    |
| Sensor Model       | $P(E_t \mid X_t)$     | Observation generation        |

State variables are usually hidden because sensors provide noisy or incomplete information rather than direct access to the true world.

---

### The Markov Assumption and Stationarity

Without simplification, the current state could depend on all past states. The **first-order Markov assumption** states:

$P(X_t \mid X_{0:t-1}) = P(X_t \mid X_{t-1})$

The present summarizes the past.

A **stationary process** assumes transition and sensor models remain constant over time.

---

### Mathematical Intuition: Joint Distribution Over Time

The temporal joint distribution factorizes as:

$P(X_{0:t}, E_{1:t}) = P(X_0)\prod_{i=1}^{t} P(X_i \mid X_{i-1}) P(E_i \mid X_i)$

Where:

* $X_{0:t}$ is the state sequence,
* $P(X_0)$ is the initial belief,
* each factor performs one transition + observation update.

This factorization enables recursive updates rather than recomputing entire histories.

---

## 2. Core Inference Tasks: Processing the Data Stream

### Types of Temporal Inference

1. **Filtering:** current belief
   $P(X_t \mid e_{1:t})$

2. **Prediction:** future belief
   $P(X_{t+k} \mid e_{1:t})$

3. **Smoothing:** past belief using future evidence
   $P(X_k \mid e_{1:t})$

4. **Most Likely Explanation:** most probable state sequence.

---

### Recursive Filtering (Forward Update)

Filtering updates the belief recursively:

$P(X_t \mid e_{1:t}) = \alpha , P(e_t \mid X_t)\sum_{x_{t-1}} P(X_t \mid x_{t-1})P(x_{t-1} \mid e_{1:t-1})$

Interpretation:

* **Prediction (sum):** propagate previous belief forward.
* **Correction (product):** incorporate current evidence.
* **Normalization $\alpha$:** ensures probabilities sum to 1.

Prediction spreads uncertainty; observation contracts it.

---

### Continuous Filtering Problem

In discrete models, beliefs are finite vectors whose representation size stays fixed.

In continuous or hybrid systems, beliefs become full probability distributions. Repeated prediction and correction steps typically make these distributions more complex.

Over time:

* representation size grows,
* mixtures multiply,
* exact filtering becomes infeasible.

This is why filtering in continuous or hybrid models can generate distributions whose representation grows without bound.

---

## 3. Hidden Markov Models (HMMs): The Discrete State Engine

HMMs assume:

* a single discrete hidden state variable,
* one evidence variable generated from that state.

### Components

* Transition matrix
  $T_{ij} = P(X_t=j \mid X_{t-1}=i)$

* Emission model
  $P(E_t \mid X_t)$

* Initial distribution
  $P(X_0)$

### Key Algorithms

* Forward algorithm (filtering)
* Backward algorithm
* Forward–Backward (smoothing)
* Viterbi algorithm (most likely sequence)

Dynamic programming allows efficient reuse of computations.

---

## 4. Kalman Filters: Continuous-State Filtering

Kalman filters handle continuous states such as position and velocity.

### Linear–Gaussian Assumptions

* transitions are linear,
* noise is Gaussian.

A Gaussian belief is fully described by:

* **Mean** — best estimate,
* **Covariance** — uncertainty.

Linear transformations preserve Gaussian form, so the belief remains a single Gaussian over time.

---

### Prediction–Correction Cycle

1. **Prediction:** uncertainty grows.
2. **Correction:** observations reduce uncertainty.

### Kalman Gain

The Kalman Gain determines how much to trust prediction vs measurement:

* accurate sensors → trust measurements,
* noisy sensors → trust predictions.

---

### Kalman Filters as a Special Case of DBNs

Every Kalman filter can be represented as a DBN with continuous variables and linear–Gaussian conditional distributions.

However:

> Not every DBN can be represented as a Kalman filter.

A Kalman filter always represents belief as a **single Gaussian bump**, while DBNs can represent arbitrary distributions.

Real-world uncertainty is often multimodal (e.g., keys may be in several distinct locations). A single Gaussian cannot model separated possibilities without assigning probability to unrealistic locations.

DBNs allow combinations of discrete and continuous variables, enabling modeling of nonlinear real-world systems.

---

## 5. Dynamic Bayesian Networks (DBNs): The General Framework

DBNs generalize HMMs and Kalman filters by using factored state representations over time.

### Structural Components

* Two-Slice Temporal Bayesian Network (2TBN)
* Factored state variables
* Local temporal dependencies

A DBN is essentially a Bayesian network unrolled through time.

---

### Relationship Between HMMs and DBNs

* Every HMM can be represented as a DBN with a single state and evidence variable.
* Any discrete DBN can be converted into an HMM by combining all state variables into one mega-state.

Mathematically equivalent — computationally very different.

DBNs exploit **sparseness**, allowing local dependencies instead of exponential global tables.

---

### 5.1 Complexity Comparison: HMM vs DBN

If a system has:

* $n$ variables,
* each with $d$ values.

#### HMM Complexity

Transition matrix size:

$O(d^{2n})$

because all global states must be enumerated.

---

#### DBN Complexity

With at most $k$ parents per variable:

$O(nd^{k+n})$

Local structure dramatically reduces required parameters.

---

#### Exact Inference Complexity

Even with DBNs, variable elimination can produce factors of size:

$O(d^{n+k})$

Exact inference remains exponential in difficult cases.

---

### 5.2 Particle Filtering (Approximate Inference)

Particle filtering approximates:

$P(X_t \mid e_{1:t})$

using $N$ samples (particles).

#### Step 1 — Propagate (Prediction)

$N(x_{t+1} \mid e_{1:t}) = \sum_{x_t} P(x_{t+1} \mid x_t)N(x_t \mid e_{1:t})$

Push particles forward via dynamics.

---

#### Step 2 — Weight (Update)

$W(x_{t+1} \mid e_{1:t+1}) = P(e_{t+1} \mid x_{t+1})N(x_{t+1} \mid e_{1:t})$

Weight particles by observation likelihood.

---

#### Step 3 — Resample

$\frac{N(x_{t+1} \mid e_{1:t+1})}{N} = \alpha W(x_{t+1} \mid e_{1:t+1})$

Good hypotheses replicate; bad ones disappear.

---

### 5.3 Consistency of Particle Filtering

As $N \to \infty$, particle filtering converges to exact filtering:

$\frac{N(x_{t+1} \mid e_{1:t+1})}{N} = \alpha P(e_{t+1} \mid x_{t+1})\sum_{x_t} P(x_{t+1} \mid x_t) P(x_t \mid e_{1:t})$

which matches:

$P(X_{t+1} \mid e_{1:t+1}) = \alpha P(e_{t+1} \mid X_{t+1})\sum_{x_t} P(X_{t+1} \mid x_t) P(x_t \mid e_{1:t})$

---

### 5.4 Diversity Collapse

Particle filtering fails when probabilities become deterministic.

For $n = 42$ binary variables:

$2^{-42} \approx 2 \times 10^{-13}$

chance of guessing the correct world initially.

If dynamics are deterministic, wrong particles remain wrong, causing collapse to a single incorrect hypothesis.

---

### 5.5 Rao–Blackwellization

To prevent collapse (e.g., SLAM), exploit conditional independence:

$P(Map \mid Evidence, Location_{1:t}) = \prod_i P(Map_i \mid Evidence, Location_{1:t})$

Strategy:

* sample only locations,
* compute map variables exactly.

This improves efficiency and accuracy.

---

## 6. Professor’s Conceptual Pitfalls & Intuition Checks

### Static Mindset Trap

Time steps are not independent; previous belief carries history.

### Belief State Misconception

Belief is a distribution, not a single value.

### Uncertainty Propagation

Without observations, uncertainty grows.

### Markov Assumption Reality Check

The Markov property holds only if the chosen state contains sufficient information.

### Continuous Filtering Insight

General continuous filtering becomes hard because belief complexity grows. Kalman filters avoid this via Gaussian assumptions.

---

## Ultra-Compressed Flow-Graph Quick Reference

### Temporal Pipeline

Environment Percepts → Markov Assumption → Prediction → Correction → Belief Distribution

---

### Algorithm Selection

* Discrete states + current belief → Forward algorithm
* Discrete states + most likely path → Viterbi
* Continuous linear–Gaussian → Kalman filter
* Complex structured systems → DBN
* Past reconstruction → Forward–Backward smoothing

---

### Key Equations

First-order Markov assumption:

$P(X_t \mid X_{0:t-1}) = P(X_t \mid X_{t-1})$

Sensor model:

$P(E_t \mid X_{0:t}, E_{0:t-1}) = P(E_t \mid X_t)$

Recursive filtering:

$f_{1:t} = \text{Forward}(f_{1:t-1}, e_t)$

Current belief depends only on previous belief and new evidence.
